## **Atelier Pandas** 

**Introduction**

Cet atelier Pandas prolonge le travail effectué avec NumPy en abordant l'analyse de données de capteurs IoT à un niveau plus proche de la réalité professionnelle, avec des données étiquetées, tabulaires et potentiellement imparfaites.

L'atelier démarre par les deux structures fondamentales de Pandas : la Series (**Partie 1**) puis le DataFrame (**Partie 2**), avant d'apprendre à explorer un jeu de données réel importé depuis un CSV (**Partie 3**). Les parties 4 et 5 couvrent la sélection de données (`loc`, `iloc`) et la manipulation de colonnes — ajout, transformation, renommage, suppression. Le filtrage (**Partie 6**) et le tri (**Partie 7**) permettent ensuite d'isoler et d'organiser les mesures pertinentes, tandis que la **Partie 8** entre dans l'analyse à proprement parler, avec des agrégations par bâtiment (`groupby`) pour identifier les zones les plus énergivores ou les plus sujettes aux alertes.

Les parties 9 et 10 traitent un aspect essentiel en conditions réelles : la qualité des données, avec la détection et le traitement des valeurs manquantes et des doublons — étape indispensable avant toute analyse fiable. L'atelier se termine par une synthèse statistique globale (**Partie 11**) et l'exportation du jeu de données nettoyé aux formats CSV et JSON (**Partie 12**) et Bonus (**Partie 13**), prêt à être transmis à un pipeline de Machine Learning.

Chaque partie se conclut par un tableau récapitulatif des fonctions utilisées et leurs alternatives, avec des explications ponctuelles sur les résultats clés et les choix méthodologiques (comme le choix de la médiane plutôt que la moyenne pour imputer la consommation).

In [7]:
# Import des bibliothèques nécessaires pour tout le notebook
import pandas as pd
import numpy as np

# On fi
# xe une graine aléatoire globale pour la reproductibilité de l'ensemble du notebook
np.random.seed(42)

print("Pandas version :", pd.__version__)
print("NumPy version  :", np.__version__)

Pandas version : 3.0.5
NumPy version  : 2.5.2


## Partie 1 – Series 

1) Création d'une Series contenant 4 températures 

In [8]:
# Une Series contenant 4 températures 
temperatures = pd.Series([24.5, 25.1, 26.3, 27.0])
print(temperatures)

0    24.5
1    25.1
2    26.3
3    27.0
dtype: float64


2) Affichage de l'index 

In [9]:
# L'Index
print(temperatures.index)

RangeIndex(start=0, stop=4, step=1)


3) Affiche la 1ère valeur 

In [10]:
# 1er valeur
print(temperatures.iloc[0])

24.5


4) Affiche la dernière valeur 

In [11]:
# derniere valeur
print(temperatures.iloc[-1])

27.0


5) Création, de deux façon différentes, une Series de 4 températures avec comme index les heures de 
mesure 12h, 13h, 14h et 15h

In [12]:
# 1 façon : passer directement le paramètre index à la création
temperatures_horaires_v1 = pd.Series(
    [24.5, 25.1, 26.3, 27.0],
    index=["12h", "13h", "14h", "15h"]
)
print("Version 1 (paramètre index) :")
print(temperatures_horaires_v1)

Version 1 (paramètre index) :
12h    24.5
13h    25.1
14h    26.3
15h    27.0
dtype: float64


In [13]:
# 2 façon : passer un dictionnaire, dont les clés deviennent l'index
temperatures_horaires_v2 = pd.Series(
    {"12h": 24.5, "13h": 25.1, "14h": 26.3, "15h": 27.0}
)
print("Version 2 (dictionnaire) :")
print(temperatures_horaires_v2)

Version 2 (dictionnaire) :
12h    24.5
13h    25.1
14h    26.3
15h    27.0
dtype: float64


 **Explication**

Les deux Series `temperatures_horaires_v1` et `temperatures_horaires_v2` contiennent exactement les mêmes valeurs et le même index (les heures). La première méthode passe l'index directement via l'argument `index=` lors de la création ; la seconde construit la Series à partir d'un dictionnaire, dont les clés deviennent automatiquement l'index. Les deux syntaxes donnent un résultat équivalent : le choix dépend surtout de la forme sous laquelle les données sont déjà disponibles.

6) Affiche la 1ère et la dernière valeur 

In [14]:
# La 1er et la dernière valeur
print("La premiere valeur est :", temperatures_horaires_v1.iloc[0], 
      "\n la dernier valeur est :", temperatures_horaires_v1.iloc[-1])   

La premiere valeur est : 24.5 
 la dernier valeur est : 27.0


###  Résumé — Partie 1 : Series

| Fonction / Méthode utilisée | Ce qu'elle fait | Alternative existante (non utilisée) | Différence avec l'alternative |
|---|---|---|---|
| `pd.Series(liste)` | Crée une Series (tableau 1D indexé) | `np.array(liste)` | `Series` ajoute un index explicite et des méta-données (nom, dtype) qu'un simple `ndarray` NumPy n'a pas — niveau d'abstraction supérieur, orienté données étiquetées. |
| `.index` | Affiche l'index (étiquettes des lignes) | `.keys()` | `Series.keys()` est un alias strict de `.index` : comportement identique. |
| `.iloc[i]` | Accès par position numérique | `.iat[i]` | `.iat` est optimisé pour accéder à un seul scalaire (plus rapide que `iloc` pour un accès unique), mais ne permet pas le slicing contrairement à `iloc`. |
| `pd.Series(data, index=[...])` | Index passé explicitement à la création | `pd.Series(dictionnaire)` | Avec un dictionnaire, les clés deviennent automatiquement l'index. Résultat équivalent ; la syntaxe par dictionnaire est plus naturelle quand chaque valeur est déjà associée à une étiquette. |


## Partie 2 – DataFrame

1) Création et affichage du dataframe df_test à partir du dictionnaire suivant 
- data = { 
"capteur": ["C001", "C002", "C003"], 
"temperature": [24.5, 25.1, 26.3], 
"humidite": [65, 67, 70] 
} 

In [15]:
# dictionnaire
data = {
    "capteur": ["C001", "C002", "C003"],
    "temperature": [24.5, 25.1, 26.3],
    "humidite": [65, 67, 70]
}

# dataframe df_test
df_test = pd.DataFrame(data)
df_test

,capteur,temperature,humidite
0,C001,24.5,65
1,C002,25.1,67
2,C003,26.3,70


2) Importation du dataset mesures_capteurs.csv dans le dataframe df 

In [16]:
# importation du dataset
df = pd.read_csv("../data/mesures_capteurs.csv")
df.head()

,id_mesure,date_heure,id_capteur,batiment,temperature,humidite,pression,consommation,etat
0,M0413,2026-01-22 04:00:00,C005,B002,25.46,58.06,1008.95,287.28,OK
1,M0290,2026-01-17 01:00:00,C002,B001,24.00,79.73,993.39,116.20,OK
2,M0077,2026-01-08 04:00:00,C005,B002,25.82,54.47,1010.32,288.50,OK
3,M0079,2026-01-08 06:00:00,C007,B003,28.23,69.39,1019.62,136.65,OK
4,M0183,2026-01-12 14:00:00,C003,B001,20.58,53.80,1016.58,182.62,OK


3) Affichage des dimensions de df 

In [17]:
# Les dimensions df
print("Dimensions de df (lignes, colonnes) :", df.shape)

Dimensions de df (lignes, colonnes) : (605, 9)


###  Résumé — Partie 2 : DataFrame

| Fonction / Méthode utilisée | Ce qu'elle fait | Alternative existante (non utilisée) | Différence avec l'alternative |
|---|---|---|---|
| `pd.DataFrame(dict)` | Crée un DataFrame à partir d'un dictionnaire (clés → colonnes) | `pd.DataFrame.from_dict(dict)` | Résultat identique ici ; `from_dict` propose en plus `orient='index'` pour construire le DataFrame à partir de dictionnaires où les clés doivent devenir des lignes plutôt que des colonnes. |
| `pd.read_csv()` | Importe un fichier CSV dans un DataFrame | `pd.read_table(path, sep=',')` | `read_table` est plus générique (accepte n'importe quel séparateur) ; `read_csv` est une version pré-configurée avec `sep=','` par défaut — résultat identique pour un CSV standard. |
| `df.shape` | Dimensions du DataFrame (lignes, colonnes) | `len(df)` + `len(df.columns)` | Résultat équivalent, mais nécessite deux appels séparés au lieu d'un seul tuple renvoyé par `shape`. |


## Partie 3 – Exploration


1) Affichage des premières lignes

In [18]:
# Les 1 lignes
df.head()

,id_mesure,date_heure,id_capteur,batiment,temperature,humidite,pression,consommation,etat
0,M0413,2026-01-22 04:00:00,C005,B002,25.46,58.06,1008.95,287.28,OK
1,M0290,2026-01-17 01:00:00,C002,B001,24.00,79.73,993.39,116.20,OK
2,M0077,2026-01-08 04:00:00,C005,B002,25.82,54.47,1010.32,288.50,OK
3,M0079,2026-01-08 06:00:00,C007,B003,28.23,69.39,1019.62,136.65,OK
4,M0183,2026-01-12 14:00:00,C003,B001,20.58,53.80,1016.58,182.62,OK


2) Affichage des 10 premières lignes 

In [19]:
# les 10 premières lignes
df.head(10)

,id_mesure,date_heure,id_capteur,batiment,temperature,humidite,pression,consommation,etat
0,M0413,2026-01-22 04:00:00,C005,B002,25.46,58.06,1008.95,287.28,OK
1,M0290,2026-01-17 01:00:00,C002,B001,24.00,79.73,993.39,116.20,OK
2,M0077,2026-01-08 04:00:00,C005,B002,25.82,54.47,1010.32,288.50,OK
3,M0079,2026-01-08 06:00:00,C007,B003,28.23,69.39,1019.62,136.65,OK
4,M0183,2026-01-12 14:00:00,C003,B001,20.58,53.80,1016.58,182.62,OK
5,M0595,2026-01-29 18:00:00,C007,B003,24.51,58.01,1013.34,146.10,OK
6,M0011,2026-01-05 10:00:00,C011,B004,29.23,70.43,1007.68,290.45,OK
7,M0132,2026-01-10 11:00:00,C012,B004,31.21,60.57,1024.64,285.92,OK
8,M0444,2026-01-23 11:00:00,C012,B004,23.11,82.06,1021.96,293.36,OK
9,M0087,2026-01-08 14:00:00,C003,B001,28.85,44.37,1008.27,206.59,OK


3) Affichage des dernières lignes 

In [20]:
# Les dernières lignes
df.tail()

,id_mesure,date_heure,id_capteur,batiment,temperature,humidite,pression,consommation,etat
600,M0072,2026-01-07 23:00:00,C012,B004,23.67,60.28,1015.11,282.36,OK
601,M0107,2026-01-09 10:00:00,C011,B004,24.34,61.42,1019.43,357.43,OK
602,M0271,2026-01-16 06:00:00,C007,B003,23.73,63.49,1016.46,152.79,OK
603,M0436,2026-01-23 03:00:00,C004,B002,24.83,79.77,1017.05,170.97,OK
604,M0103,2026-01-09 06:00:00,C007,B003,17.02,52.04,1001.14,44.99,OK


4) Affiche le nombre de lignes

In [21]:
# Nombre de lignes
print("nombre de ligne est :", df.shape[0])

nombre de ligne est : 605


5) Affiche le nombre de colonnes

In [22]:
# Nombre de colonnes 
print("nombre de colonne est :", df.shape[1])

nombre de colonne est : 9


6) Affiche les noms des colonnes 

In [23]:
# noms des colonnes
print(df.columns.tolist())

['id_mesure', 'date_heure', 'id_capteur', 'batiment', 'temperature', 'humidite', 'pression', 'consommation', 'etat']


7) Affiche les informations générales 

In [24]:
# Les informations generales
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 605 entries, 0 to 604
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id_mesure     605 non-null    str    
 1   date_heure    605 non-null    str    
 2   id_capteur    605 non-null    str    
 3   batiment      605 non-null    str    
 4   temperature   599 non-null    float64
 5   humidite      600 non-null    float64
 6   pression      600 non-null    float64
 7   consommation  600 non-null    float64
 8   etat          601 non-null    str    
dtypes: float64(4), str(5)
memory usage: 42.7 KB


 **Explication**

`df.info()` affiche, pour chaque colonne : son nom, le nombre de valeurs non nulles (*non-null count*) et son type de données (*dtype*). C'est la première chose à vérifier après un import : cela permet de repérer rapidement d'éventuelles valeurs manquantes (colonne avec moins de non-null que le nombre total de lignes) ou des types inattendus (par exemple une colonne numérique importée comme texte). La dernière ligne indique aussi l'usage mémoire approximatif du DataFrame.

8) Affiche les statistiques descriptives

In [25]:
df.describe()

,temperature,humidite,pression,consommation
count,599.000000,600.00000,600.000000,600.000000
mean,24.878314,64.92620,1012.221900,208.675417
std,4.059576,10.76905,10.599042,72.243567
min,-18.500000,28.52000,850.000000,18.120000
25%,22.570000,58.17250,1006.790000,160.177500
50%,24.860000,65.37500,1012.855000,206.150000
75%,27.275000,71.61500,1017.827500,254.127500
max,58.700000,145.00000,1038.430000,875.000000


 **Explication**

`df.describe()` calcule automatiquement, pour chaque colonne **numérique**, le nombre de valeurs (`count`), la moyenne (`mean`), l'écart-type (`std`), le minimum, les quartiles (25 %, 50 % = médiane, 75 %) et le maximum. Cela donne un premier aperçu de la distribution des données : si le maximum est très éloigné du 75ᵉ percentile, cela peut indiquer la présence de valeurs extrêmes (outliers) à surveiller. Les colonnes textuelles/catégorielles (`capteur`, `batiment`, `etat`) sont ignorées par défaut, car ces statistiques ne s'appliquent qu'à des valeurs numériques.

###  Résumé — Partie 3 : Exploration

| Fonction / Méthode utilisée | Ce qu'elle fait | Alternative existante (non utilisée) | Différence avec l'alternative |
|---|---|---|---|
| `df.head(n)` | Affiche les n premières lignes (5 par défaut) | `df[:n]` | Slicing équivalent pour les lignes ; `head()` reste plus explicite et lisible. |
| `df.tail()` | Affiche les dernières lignes | `df[-5:]` | Slicing équivalent ; `tail()` est plus lisible et explicite sur l'intention. |
| `df.shape[0]` / `df.shape[1]` | Nombre de lignes / colonnes | `len(df)` / `len(df.columns)` | `len(df)` est l'équivalent Python classique pour compter les lignes ; résultat identique à `shape[0]`. |
| `df.columns.tolist()` | Liste des noms de colonnes | `list(df.columns)` | Résultat strictement identique, deux syntaxes équivalentes. |
| `df.info()` | Résumé technique : types, valeurs non nulles, mémoire | `df.dtypes` | `dtypes` ne donne que les types de données, sans le nombre de valeurs non manquantes ni l'usage mémoire fournis par `info()`. |
| `df.describe()` | Statistiques descriptives des colonnes numériques | `df.describe(include='all')` | Sans argument, `describe()` ignore les colonnes non numériques ; `include='all'` ajoute des statistiques pour les colonnes catégorielles (valeur la plus fréquente, nombre de catégories uniques…). |


## Partie 4 – Sélection

1) Sélection de la colonne "temperature"

In [26]:
# Selection de la colonne tmp
df["temperature"]

0      25.46
1      24.00
2      25.82
3      28.23
4      20.58
       ...  
600    23.67
601    24.34
602    23.73
603    24.83
604    17.02
Name: temperature, Length: 605, dtype: float64

2) Sélection des colonnes "temperature", "humidite", "pression" et "consommation"  


In [27]:
# Sélection des colonnes "temperature", "humidite", "pression" et "consommation"  
df[["temperature", "humidite", "pression", "consommation"]]

,temperature,humidite,pression,consommation
0,25.46,58.06,1008.95,287.28
1,24.00,79.73,993.39,116.20
2,25.82,54.47,1010.32,288.50
3,28.23,69.39,1019.62,136.65
4,20.58,53.80,1016.58,182.62
...,...,...,...,...
600,23.67,60.28,1015.11,282.36
601,24.34,61.42,1019.43,357.43
602,23.73,63.49,1016.46,152.79
603,24.83,79.77,1017.05,170.97


3) Sélection des 5 premières lignes avec loc 

In [28]:
# les 5 premieres lignes 
df.loc[0:4]

,id_mesure,date_heure,id_capteur,batiment,temperature,humidite,pression,consommation,etat
0,M0413,2026-01-22 04:00:00,C005,B002,25.46,58.06,1008.95,287.28,OK
1,M0290,2026-01-17 01:00:00,C002,B001,24.00,79.73,993.39,116.20,OK
2,M0077,2026-01-08 04:00:00,C005,B002,25.82,54.47,1010.32,288.50,OK
3,M0079,2026-01-08 06:00:00,C007,B003,28.23,69.39,1019.62,136.65,OK
4,M0183,2026-01-12 14:00:00,C003,B001,20.58,53.80,1016.58,182.62,OK


4) Sélection des 5 premières lignes et les 4 premières colonnes avec iloc

In [29]:
# les 5 premieres lignes et les 4 premiere colonnes
df.iloc[0:5, 0:4]

,id_mesure,date_heure,id_capteur,batiment
0,M0413,2026-01-22 04:00:00,C005,B002
1,M0290,2026-01-17 01:00:00,C002,B001
2,M0077,2026-01-08 04:00:00,C005,B002
3,M0079,2026-01-08 06:00:00,C007,B003
4,M0183,2026-01-12 14:00:00,C003,B001


###  Résumé — Partie 4 : Sélection

| Fonction / Méthode utilisée | Ce qu'elle fait | Alternative existante (non utilisée) | Différence avec l'alternative |
|---|---|---|---|
| `df["colonne"]` | Sélectionne une colonne (renvoie une Series) | `df.colonne` (notation attribut) | Résultat identique si le nom de colonne est un identifiant Python valide (sans espace ni caractère spécial) ; `df["col"]` fonctionne, elle, dans tous les cas. |
| `df[["c1","c2"]]` | Sélectionne plusieurs colonnes (renvoie un DataFrame) | `df.loc[:, ["c1","c2"]]` | Résultat identique ; `.loc` permet en plus de combiner cette sélection de colonnes avec un filtrage de lignes en une seule instruction. |
| `df.loc[0:4]` | Sélection par étiquettes (bornes incluses) | `df.iloc[0:5]` | Avec un index numérique par défaut, `loc[0:4]` inclut la borne 4, alors que `iloc[0:5]` (position) l'exclut — il faut donc `iloc[0:5]` pour obtenir les 5 mêmes lignes que `loc[0:4]`. |
| `df.iloc[0:5, 0:4]` | Sélection par position (lignes et colonnes) | `df.loc[df.index[0:5], df.columns[0:4]]` | Résultat équivalent mais nécessite de convertir les positions en étiquettes manuellement ; `iloc` est plus direct pour une sélection purement positionnelle. |


## Partie 5 – Manipulation des colonnes 

1) Ajoute de la colonne "temperature_fahrenheit"

In [30]:
# ajoute de la colonne
df["temperature_fahrenheit"] = df["temperature"] * 9 / 5 + 32
df[["temperature", "temperature_fahrenheit"]].head()

,temperature,temperature_fahrenheit
0,25.46,77.828
1,24.00,75.200
2,25.82,78.476
3,28.23,82.814
4,20.58,69.044


2) Ajoutons la colonne "niveau_temperature" pour catégoriser la température en  
"Élevée" 
- (température > 30) ou "Normale". On peut utiliser la méthode where() de numpy 

In [31]:
df["niveau_temperature"] = np.where(df["temperature"] > 30, "Élevée", "Normale")
df[["temperature", "niveau_temperature"]].head()

,temperature,niveau_temperature
0,25.46,Normale
1,24.00,Normale
2,25.82,Normale
3,28.23,Normale
4,20.58,Normale


In [32]:
df["niveau_temperature"].value_counts()

niveau_temperature
Normale    559
Élevée      46
Name: count, dtype: int64

**Explication**


Le résultat de `value_counts()` indique combien de mesures ont été classées « Élevée » (température > 30 °C) contre « Normale ». Ce comptage donne une première idée de la proportion de situations potentiellement à risque dans le jeu de données, avant même d'aller plus loin dans l'analyse par bâtiment.

3) Renommage de la colonne "humidite" en "humidite_relative"

In [33]:
# renommage de la colonne 
df = df.rename(columns={"humidite": "humidite_relative"})
df.columns.tolist()

['id_mesure',
 'date_heure',
 'id_capteur',
 'batiment',
 'temperature',
 'humidite_relative',
 'pression',
 'consommation',
 'etat',
 'temperature_fahrenheit',
 'niveau_temperature']

4) Supprime de la colonne "temperature_fahrenheit"

In [34]:
# colonne supprimer et nouveau visualisation
df = df.drop(columns=["temperature_fahrenheit"])
df.columns.tolist()

['id_mesure',
 'date_heure',
 'id_capteur',
 'batiment',
 'temperature',
 'humidite_relative',
 'pression',
 'consommation',
 'etat',
 'niveau_temperature']

###  Résumé — Partie 5 : Manipulation des colonnes

| Fonction / Méthode utilisée | Ce qu'elle fait | Alternative existante (non utilisée) | Différence avec l'alternative |
|---|---|---|---|
| `df["nouvelle_col"] = ...` | Ajoute/calcule une colonne (vectorisé) | `df.assign(nouvelle_col=...)` | `assign()` renvoie un **nouveau** DataFrame sans modifier l'original (utile pour chaîner des opérations), alors que l'affectation directe modifie `df` en place. |
| `np.where(cond, val_vrai, val_faux)` | Crée une colonne catégorielle selon une condition | `df["col"].apply(lambda x: ...)` | `np.where` est vectorisé et bien plus rapide sur de grands DataFrames ; `apply()` applique une fonction Python ligne par ligne, donc plus lent. |
| `.value_counts()` | Compte les occurrences de chaque valeur unique | `df.groupby("col").size()` | Résultat équivalent (mêmes comptages), mais `value_counts()` trie automatiquement par fréquence décroissante, contrairement à `groupby().size()`. |
| `df.rename(columns={...})` | Renomme une ou plusieurs colonnes | `df.columns = [...]` | `rename()` cible précisément les colonnes à changer par leur nom (plus sûr) ; réaffecter `df.columns` oblige à lister TOUTES les colonnes dans le bon ordre. |
| `df.drop(columns=[...])` | Supprime une ou plusieurs colonnes | `del df["colonne"]` | `del` supprime une seule colonne à la fois et modifie `df` en place ; `drop()` peut en supprimer plusieurs à la fois et renvoie une copie par défaut (sauf `inplace=True`). |


## Partie 6 – Filtrage 

1) Récupération uniquement les lignes telles que température > 30 

In [35]:
# les lignes sont recuperer si tmp > 30
df_temp_elevee = df[df["temperature"] > 30]
print("Nombre de lignes :", df_temp_elevee.shape[0])
df_temp_elevee.head()

Nombre de lignes : 46


,id_mesure,date_heure,id_capteur,batiment,temperature,humidite_relative,pression,consommation,etat,niveau_temperature
7,M0132,2026-01-10 11:00:00,C012,B004,31.21,60.57,1024.64,285.92,OK,Élevée
43,M0575,2026-01-28 22:00:00,C011,B004,34.72,57.50,1008.94,266.22,ALERTE,Élevée
47,M0082,2026-01-08 09:00:00,C010,B004,32.46,72.31,1000.42,276.99,ALERTE,Élevée
63,M0064,2026-01-07 15:00:00,C004,B002,31.19,55.13,1011.04,264.98,OK,Élevée
79,M0275,2026-01-16 10:00:00,C011,B004,31.74,69.95,1020.79,335.89,OK,Élevée


2) Récupération uniquement les lignes telles température > 30 et humidité > 70 

In [36]:
# les lignes telles température > 30 et humidité > 70 
df_temp_hum_elevee = df[(df["temperature"] > 30) & (df["humidite_relative"] > 70)]
print("Nombre de lignes :", df_temp_hum_elevee.shape[0])
df_temp_hum_elevee.head()

Nombre de lignes : 14


,id_mesure,date_heure,id_capteur,batiment,temperature,humidite_relative,pression,consommation,etat,niveau_temperature
47,M0082,2026-01-08 09:00:00,C010,B004,32.46,72.31,1000.42,276.99,ALERTE,Élevée
88,M0288,2026-01-16 23:00:00,C012,B004,30.06,75.30,1014.47,323.32,OK,Élevée
129,M0592,2026-01-29 15:00:00,C004,B002,33.60,79.80,1021.96,212.55,ALERTE,Élevée
145,M0503,2026-01-25 22:00:00,C011,B004,30.57,71.26,1002.88,359.97,OK,Élevée
165,M0323,2026-01-18 10:00:00,C011,B004,33.01,73.24,1014.36,260.24,ALERTE,Élevée


###  Résumé — Partie 6 : Filtrage

| Fonction / Méthode utilisée | Ce qu'elle fait | Alternative existante (non utilisée) | Différence avec l'alternative |
|---|---|---|---|
| `df[df["col"] > x]` | Filtre les lignes selon une condition booléenne | `df.query("col > x")` | Résultat identique ; `query()` utilise une syntaxe en chaîne de caractères, souvent plus lisible pour des conditions complexes, mais légèrement plus lente sur de très grands DataFrames. |
| `(cond1) & (cond2)` | Combine plusieurs conditions | `df.query("cond1 and cond2")` | Résultat identique à `&`, mais `query()` permet d'écrire les conditions en langage plus naturel (`and`/`or`) sans parenthèses ni opérateurs bit à bit. |


## Partie 7 : Tri 

1) Création df_trie1 pour trier en fonction de la température croissante

In [37]:
#  df_trie1 pour trier en fonction de la température croissante
df_trie1 = df.sort_values(by="temperature", ascending=True)
df_trie1.head()

,id_mesure,date_heure,id_capteur,batiment,temperature,humidite_relative,pression,consommation,etat,niveau_temperature
487,M0538,2026-01-27 09:00:00,C010,B004,-18.50,66.27,1012.49,312.92,ERREUR,Normale
245,M0127,2026-01-10 06:00:00,C007,B003,14.11,145.00,1032.36,169.57,ERREUR,Normale
258,M0223,2026-01-14 06:00:00,C007,B003,15.49,47.96,1006.34,124.90,OK,Normale
249,M0175,2026-01-12 06:00:00,C007,B003,16.04,55.36,1005.68,140.95,OK,Normale
59,M0146,2026-01-11 01:00:00,C002,B001,16.77,53.44,1010.65,131.75,OK,Normale


2) Création df_trie2 pour trier en fonction de la température décroissante

In [38]:
# df_trie2 pour trier en fonction de la température décroissante
df_trie2 = df.sort_values(by="temperature", ascending=False)
df_trie2.head()

,id_mesure,date_heure,id_capteur,batiment,temperature,humidite_relative,pression,consommation,etat,niveau_temperature
502,M0048,2026-01-06 23:00:00,C012,B004,58.70,81.51,1026.79,271.92,ERREUR,Élevée
43,M0575,2026-01-28 22:00:00,C011,B004,34.72,57.50,1008.94,266.22,ALERTE,Élevée
129,M0592,2026-01-29 15:00:00,C004,B002,33.60,79.80,1021.96,212.55,ALERTE,Élevée
211,M0370,2026-01-20 09:00:00,C010,B004,33.29,50.43,1009.87,356.76,ALERTE,Élevée
165,M0323,2026-01-18 10:00:00,C011,B004,33.01,73.24,1014.36,260.24,ALERTE,Élevée


3) Création top10 pour voir les 10 températures les plus élevées 

In [39]:
# top10 pour voir les 10 températures les plus élevées 
top10 = df.nlargest(10, "temperature")
top10[["id_mesure", "batiment", "temperature"]]

,id_mesure,batiment,temperature
502,M0048,B004,58.70
43,M0575,B004,34.72
129,M0592,B002,33.60
211,M0370,B004,33.29
165,M0323,B004,33.01
118,M0365,B002,32.90
520,M0063,B001,32.72
438,M0558,B002,32.56
47,M0082,B004,32.46
169,M0546,B002,32.36


4) Trie en fonction des colonnes "batiment" (croissant) et "temperature" (décroissant)

In [40]:
df_trie_multi = df.sort_values(by=["batiment", "temperature"], ascending=[True, False])
df_trie_multi.head(10)

,id_mesure,date_heure,id_capteur,batiment,temperature,humidite_relative,pression,consommation,etat,niveau_temperature
520,M0063,2026-01-07 14:00:00,C003,B001,32.72,53.28,1010.05,195.37,ALERTE,Élevée
447,M0170,2026-01-12 01:00:00,C002,B001,31.15,69.20,1016.10,172.49,OK,Élevée
224,M0303,2026-01-17 14:00:00,C003,B001,30.84,67.98,1020.09,157.99,OK,Élevée
500,M0062,2026-01-07 13:00:00,C002,B001,30.39,52.13,1004.23,262.66,OK,Élevée
9,M0087,2026-01-08 14:00:00,C003,B001,28.85,44.37,1008.27,206.59,OK,Normale
467,M0241,2026-01-15 00:00:00,C001,B001,28.85,66.31,1004.98,175.06,OK,Normale
567,M0591,2026-01-29 14:00:00,C003,B001,28.71,55.02,1010.18,201.62,OK,Normale
85,M0074,2026-01-08 01:00:00,C002,B001,28.41,39.33,1011.11,187.94,OK,Normale
293,M0230,2026-01-14 13:00:00,C002,B001,28.31,70.29,1023.91,95.36,OK,Normale
263,M0579,2026-01-29 02:00:00,C003,B001,28.11,65.56,1016.97,196.48,OK,Normale


###  Résumé — Partie 7 : Tri

| Fonction / Méthode utilisée | Ce qu'elle fait | Alternative existante (non utilisée) | Différence avec l'alternative |
|---|---|---|---|
| `df.sort_values("col")` | Trie les lignes selon une colonne (croissant par défaut) | `df.sort_values("col", ascending=False)` | Comparaison directe : `ascending=True` (défaut) trie en croissant, `False` inverse simplement l'ordre — déjà illustré dans le notebook. |
| `df.nlargest(n, "col")` | Renvoie les n lignes ayant les plus grandes valeurs | `df.sort_values("col", ascending=False).head(n)` | Résultat identique ; `nlargest()` est optimisé en interne (plus rapide) et plus concis pour ce cas précis. |
| `df.sort_values(by=[...], ascending=[...])` | Tri multi-colonnes avec un ordre différent par colonne | Tri manuel via plusieurs `sort_values` successifs | Beaucoup plus complexe et sujet à erreur ; `sort_values` avec des listes est la méthode directe et recommandée pour un tri multi-critères. |


## Partie 8 : Analyse 

1) Affiche la consommation moyenne par bâtiment 

In [41]:
# la consommation moyenne par bâtiment 
conso_moyenne_par_batiment = df.groupby("batiment")["consommation"].mean()
conso_moyenne_par_batiment

batiment
B001    177.870927
B002    221.785333
B003    153.191000
B004    282.552282
Name: consommation, dtype: float64

2) Affiche le minimum, le maximum, la moyenne et l’écart-type de la température par bâtiment

In [42]:
# e minimum, le maximum, la moyenne et l’écart-type de la température par bâtiment
stats_temp_par_batiment = df.groupby("batiment")["temperature"].agg(["min", "max", "mean", "std"])
stats_temp_par_batiment

,min,max,mean,std
batiment,,,,
B001,16.77,32.72,23.578824,2.961211
B002,17.67,33.60,25.778000,2.951828
B003,14.11,31.74,23.013699,2.983902
B004,-18.50,58.70,27.119000,5.367361


3) Affichage, par bâtiment, le maximum et la moyenne de la température et de la consommation 

In [43]:
stats_par_batiment = df.groupby("batiment").agg({
    "temperature": ["max", "mean"],
    "consommation": ["max", "mean"]
})
stats_par_batiment

temperature            consommation            
                 max       mean          max        mean
batiment                                                
B001           32.72  23.578824       294.72  177.870927
B002           33.60  25.778000       335.20  221.785333
B003           31.74  23.013699       875.00  153.191000
B004           58.70  27.119000       411.12  282.552282

 **Explication**

`stats_par_batiment` possède désormais des colonnes **multi-niveaux** (par exemple `('temperature', 'max')`, `('temperature', 'mean')`, `('consommation', ...)`), car on a demandé plusieurs statistiques différentes pour plusieurs colonnes en une seule instruction. Pour accéder à une statistique précise, on utilise une double clé, par exemple `stats_par_batiment[('temperature', 'mean')]`.

4) Affichage de la température moyenne globale

In [44]:
print("Température moyenne globale :", round(df["temperature"].mean(), 2))

Température moyenne globale : 24.88


5) Affiche la température maximale 

In [45]:
print("Température maximale :", df["temperature"].max())

Température maximale : 58.7


6) Affiche la consommation totale

In [46]:
print("Consommation totale :", round(df["consommation"].sum(), 2))

Consommation totale : 125205.25


7) Le bâtiment consomme le plus en moyenne ?

In [47]:
batiment_plus_energivore = conso_moyenne_par_batiment.idxmax()
print("Bâtiment consommant le plus en moyenne :", batiment_plus_energivore)
print("Consommation moyenne associée :", round(conso_moyenne_par_batiment.max(), 2))

Bâtiment consommant le plus en moyenne : B004
Consommation moyenne associée : 282.55


**Explication**

`idxmax()` renvoie l'**étiquette** (ici le nom du bâtiment) associée à la valeur maximale de `conso_moyenne_par_batiment`, et non la valeur elle-même. C'est ce bâtiment qui devra être surveillé en priorité par le futur système de détection d'anomalies, puisqu'il consomme le plus d'énergie en moyenne.

8) Affiche le nombre d'alertes par bâtiment 

In [48]:
alertes_par_batiment = df[df["etat"] == "ALERTE"].groupby("batiment").size()
alertes_par_batiment

batiment
B001     2
B002     7
B003     5
B004    15
dtype: int64

 **Explication**

Ce comptage montre combien de mesures ont été enregistrées avec l'état `"ALERTE"` pour chaque bâtiment. Comparé au nombre total de mesures par bâtiment, il permettrait de calculer un taux d'alerte et d'identifier les bâtiments les plus problématiques.

###  Résumé — Partie 8 : Analyse

| Fonction / Méthode utilisée | Ce qu'elle fait | Alternative existante (non utilisée) | Différence avec l'alternative |
|---|---|---|---|
| `df.groupby("col").mean()` | Moyenne d'une variable numérique par groupe | `df.pivot_table(values=..., index="col", aggfunc="mean")` | Résultat équivalent ; `pivot_table` est plus flexible pour croiser plusieurs dimensions (lignes ET colonnes) en une seule table. |
| `df.groupby("col").agg({...})` | Applique plusieurs statistiques différentes par colonne et par groupe | `df.groupby("col").describe()` | `describe()` calcule automatiquement un ensemble standard de statistiques pour toutes les colonnes numériques, alors que `agg()` permet de choisir précisément quelles statistiques appliquer à quelles colonnes. |
| `.idxmax()` | Renvoie l'étiquette (index) de la valeur maximale | `.sort_values(ascending=False).index[0]` | Résultat identique, mais `idxmax()` est plus direct et plus lisible pour ne récupérer que le premier élément. |
| `df[condition].groupby("col").size()` | Compte les lignes par groupe après filtrage | `df[condition]["col"].value_counts()` | Résultat identique dans ce cas ; `value_counts()` est une alternative plus concise quand on ne groupe que sur une seule colonne. |


## Partie 9 : Gestion des valeurs manquantes 

1) Affiche le nombre de valeurs manquantes par colonne 

In [49]:
df.isna().sum()

id_mesure             0
date_heure            0
id_capteur            0
batiment              0
temperature           6
humidite_relative     5
pression              5
consommation          5
etat                  4
niveau_temperature    0
dtype: int64

2) Affiche le nombre total des valeurs manquantes 

In [50]:
print("Total des valeurs manquantes :", df.isna().sum().sum())

Total des valeurs manquantes : 25


3) Affiche le taux de valeurs manquantes 

In [51]:
taux_manquantes = df.isna().sum().sum() / df.size * 100
print(f"Taux de valeurs manquantes : {taux_manquantes:.2f} %")

Taux de valeurs manquantes : 0.41 %


 **Explication**

Ce taux exprime la proportion de cellules manquantes sur l'ensemble du DataFrame (toutes colonnes et toutes lignes confondues). Un taux faible (quelques pourcents) justifie généralement un remplacement (imputation) des valeurs manquantes plutôt que la suppression des lignes concernées, qui ferait perdre trop d'informations.

4) Affiche les lignes contenant des valeurs manquantes 

In [52]:
lignes_manquantes = df[df.isna().any(axis=1)]
print("Nombre de lignes concernées :", lignes_manquantes.shape[0])
lignes_manquantes

Nombre de lignes concernées : 25


,id_mesure,date_heure,id_capteur,batiment,temperature,humidite_relative,pression,consommation,etat,niveau_temperature
14,M0056,2026-01-07 07:00:00,C008,B003,22.33,67.42,1014.41,101.20,NaN,Normale
30,M0073,2026-01-08 00:00:00,C001,B001,23.12,63.97,NaN,186.87,OK,Normale
91,M0473,2026-01-24 16:00:00,C005,B002,21.32,70.43,1021.20,194.76,NaN,Normale
128,M0300,2026-01-17 11:00:00,C012,B004,23.32,55.73,NaN,232.38,OK,Normale
139,M0345,2026-01-19 08:00:00,C009,B003,NaN,66.77,1015.33,216.31,OK,Normale
146,M0118,2026-01-09 21:00:00,C010,B004,30.32,NaN,1015.78,261.68,OK,Élevée
149,M0034,2026-01-06 09:00:00,C010,B004,28.91,NaN,1001.23,234.30,OK,Normale
163,M0019,2026-01-05 18:00:00,C007,B003,NaN,69.97,1014.14,181.07,OK,Normale
196,M0372,2026-01-20 11:00:00,C012,B004,29.33,56.91,1020.02,NaN,OK,Normale
215,M0204,2026-01-13 11:00:00,C012,B004,NaN,59.96,1007.98,294.02,OK,Normale


5) Pour la température et l’humidité, remplacement les valeurs manquantes par les moyennes

In [53]:
df["temperature"] = df["temperature"].fillna(df["temperature"].mean())
df["humidite_relative"] = df["humidite_relative"].fillna(df["humidite_relative"].mean())
df[["temperature", "humidite_relative"]].isna().sum()

temperature          0
humidite_relative    0
dtype: int64

 **Explication**

La moyenne est utilisée pour `temperature` et `humidite_relative` car ce sont des mesures physiques dont la distribution est généralement assez symétrique, sans valeurs extrêmes majeures : la moyenne reste donc un bon estimateur central pour remplacer les valeurs manquantes sans trop déformer la distribution globale.

6) Pour la consommation, remplacement des valeurs manquantes par la médiane 


In [54]:
df["consommation"] = df["consommation"].fillna(df["consommation"].median())
df["consommation"].isna().sum()

np.int64(0)

 **Explication**

La médiane est préférée à la moyenne pour `consommation`, car cette variable est plus susceptible de contenir des valeurs extrêmes (pics de consommation ponctuels). Contrairement à la moyenne, la médiane n'est pas influencée par ces valeurs atypiques, ce qui en fait un choix plus robuste pour l'imputation de cette colonne.

7) Pour "etat", remplacement des valeurs manquantes par "INCONNU"

In [55]:
df["etat"] = df["etat"].fillna("INCONNU")
df["etat"].value_counts()

etat
OK         567
ALERTE      29
ERREUR       5
INCONNU      4
Name: count, dtype: int64

In [56]:
# verification 
df.isna().sum()

id_mesure             0
date_heure            0
id_capteur            0
batiment              0
temperature           0
humidite_relative     0
pression              5
consommation          0
etat                  0
niveau_temperature    0
dtype: int64

###  Résumé — Partie 9 : Gestion des valeurs manquantes

| Fonction / Méthode utilisée | Ce qu'elle fait | Alternative existante (non utilisée) | Différence avec l'alternative |
|---|---|---|---|
| `df.isna().sum()` | Nombre de valeurs manquantes par colonne | `df.isnull().sum()` | Résultat strictement identique ; `isnull()` est un alias exact de `isna()` dans pandas. |
| `df.isna().sum().sum()` | Nombre total de valeurs manquantes (tout le DataFrame) | `df.isna().values.sum()` | Résultat identique ; `.values` convertit d'abord en tableau NumPy avant de sommer, généralement un peu plus rapide sur de très grands DataFrames. |
| `df[df.isna().any(axis=1)]` | Lignes contenant au moins une valeur manquante | `df[df.isna().sum(axis=1) > 0]` | Résultat équivalent ; `any(axis=1)` s'arrête dès qu'une valeur manquante est trouvée sur la ligne (plus rapide), alors que `sum(axis=1)` compte systématiquement toutes les valeurs manquantes de la ligne. |
| `.fillna(moyenne)` | Remplace les valeurs manquantes par la moyenne de la colonne | `.fillna(method='ffill')` | `ffill` propage la dernière valeur connue vers l'avant — une approche différente, adaptée aux séries temporelles plutôt qu'à des mesures numériques indépendantes. |
| `.fillna(médiane)` | Remplace les valeurs manquantes par la médiane (robuste aux valeurs extrêmes) | `.fillna(moyenne)` | La moyenne est sensible aux valeurs extrêmes (outliers) ; la médiane reste stable même en présence de pics de consommation — d'où son choix pour cette colonne. |
| `.fillna("INCONNU")` | Remplace les valeurs manquantes d'une colonne catégorielle par une étiquette explicite | `.dropna(subset=["etat"])` | `dropna` supprimerait entièrement les lignes concernées au lieu de les conserver avec une valeur par défaut — un choix différent selon qu'on préfère garder ou écarter les données incomplètes. |


## Partie 10 : Gestion des doublons 

1) Affiche le nombre de doublons 

In [57]:
print("Nombre de doublons :", df.duplicated().sum())

Nombre de doublons : 5


2) Affiche des doublons

In [58]:
df[df.duplicated()]

,id_mesure,date_heure,id_capteur,batiment,temperature,humidite_relative,pression,consommation,etat,niveau_temperature
183,M0599,2026-01-29 22:00:00,C011,B004,22.02,68.09,1005.90,227.81,OK,Normale
231,M0026,2026-01-06 01:00:00,C002,B001,22.87,77.99,1010.15,213.19,OK,Normale
355,M0147,2026-01-11 02:00:00,C003,B001,26.14,84.97,1003.59,142.31,OK,Normale
538,M0456,2026-01-23 23:00:00,C012,B004,20.68,72.69,1022.77,309.01,OK,Normale
539,M0302,2026-01-17 13:00:00,C002,B001,22.05,58.26,1007.30,140.42,OK,Normale


3) Supprimons les doublons puis vérifier la suppression

In [59]:
df = df.drop_duplicates()
print("Nombre de doublons après suppression :", df.duplicated().sum())
print("Nouvelles dimensions de df :", df.shape)

Nombre de doublons après suppression : 0
Nouvelles dimensions de df : (600, 10)


###  Résumé — Partie 10 : Gestion des doublons

| Fonction / Méthode utilisée | Ce qu'elle fait | Alternative existante (non utilisée) | Différence avec l'alternative |
|---|---|---|---|
| `df.duplicated()` | Détecte les lignes strictement identiques à une ligne précédente | `df.duplicated(subset=["col"])` | Par défaut, `duplicated()` compare TOUTES les colonnes ; `subset` permet de ne considérer que certaines colonnes pour définir un doublon (ex. même capteur + même horodatage). |
| `df.drop_duplicates()` | Supprime les lignes dupliquées (garde la 1ère occurrence par défaut) | `df.drop_duplicates(keep="last")` | `keep="last"` conserve la dernière occurrence de chaque doublon au lieu de la première ; `keep=False` supprimerait même toutes les occurrences dupliquées. |


## Partie 11 : Statistiques descriptives

1) Pour la température, affiche le minimum, le maximum, la moyenne, la mediane, l’écart-type et 
le nombre de valeurs 

In [60]:
stats_temperature = {
    "min": df["temperature"].min(),
    "max": df["temperature"].max(),
    "mean": df["temperature"].mean(),
    "median": df["temperature"].median(),
    "std": df["temperature"].std(),
    "count": df["temperature"].count()
}
pd.Series(stats_temperature)

min       -18.500000
max        58.700000
mean       24.896033
median     24.878314
std         4.048027
count     600.000000
dtype: float64

2) Pour "etat", affiche le mode et le nombre de mesures par état (OK, ALERTE, ERREUR) 

In [61]:
print("Mode de la colonne 'etat' :", df["etat"].mode()[0])
print()
print(df["etat"].value_counts())

Mode de la colonne 'etat' : OK

etat
OK         562
ALERTE      29
ERREUR       5
INCONNU      4
Name: count, dtype: int64


 **Explication**

Le mode indique l'état le plus fréquemment observé parmi les capteurs (généralement `"OK"` si le système fonctionne majoritairement bien). Le détail de `value_counts()` permet ensuite de voir la répartition complète entre les états `OK`, `ALERTE`, `ERREUR` et `INCONNU` (les valeurs précédemment manquantes).

###  Résumé — Partie 11 : Statistiques descriptives

| Fonction / Méthode utilisée | Ce qu'elle fait | Alternative existante (non utilisée) | Différence avec l'alternative |
|---|---|---|---|
| Dictionnaire manuel {min, max, mean, median, std} | Rassemble plusieurs statistiques dans une structure personnalisée | `df["temperature"].describe()` | `describe()` calcule automatiquement un ensemble standard de statistiques (count, mean, std, min, quartiles, max) en une seule instruction, sans avoir à les assembler manuellement. |
| `.mode()[0]` | Valeur la plus fréquente d'une colonne catégorielle | `.value_counts().idxmax()` | Résultat identique ; `value_counts().idxmax()` est une alternative plus verbeuse qui montre au passage les comptages de toutes les catégories. |
| `.value_counts()` | Nombre d'occurrences de chaque catégorie | `df.groupby("col").size()` | Résultat équivalent (déjà comparé en Partie 5) ; `value_counts()` trie par défaut par fréquence décroissante. |


## Partie 12 : Exportation 

1) Exportation du dataframe nettoyé dans exports/donnees_nettoyees.csv 

In [62]:
df.to_csv("../exports/donnees_nettoyees.csv", index=False)
print("Export CSV terminé.")

Export CSV terminé.


2) Exportation du dataframe nettoyé dans exports/donnees_nettoyees.json

In [63]:
df.to_json("../exports/donnees_nettoyees.json", orient="records", indent=2, force_ascii=False)
print("Export JSON terminé.")

Export JSON terminé.


 **Explication**

`orient="records"` produit une liste d'objets JSON, un par ligne du DataFrame (le format le plus courant pour un usage en API ou en JavaScript) ; `indent=2` rend le fichier lisible par un humain ; `force_ascii=False` conserve les accents français au lieu de les encoder en séquences unicode illisibles (comme `\u00e9`).

###  Résumé — Partie 12 : Exportation

| Fonction / Méthode utilisée | Ce qu'elle fait | Alternative existante (non utilisée) | Différence avec l'alternative |
|---|---|---|---|
| `df.to_csv(path, index=False)` | Exporte le DataFrame au format CSV | `df.to_excel(path, index=False)` | `to_excel` produit un fichier `.xlsx` (nécessite le package `openpyxl`), utile pour un usage bureautique plutôt qu'un format texte universel comme le CSV. |
| `df.to_json(path, orient="records", indent=2, force_ascii=False)` | Exporte au format JSON, une entrée par enregistrement | `json.dump(df.to_dict(orient="records"), f)` | Résultat équivalent en combinant deux étapes manuelles ; `to_json()` fait tout en une seule instruction et gère nativement l'encodage, l'indentation et les types pandas/numpy. |


## Partie 13 — BONUS : Analyse Temporelle & Détection Automatique des Anomalies

Dans un contexte réel de traitement de données IoT (capteurs) :  
- Qualité des données (Data Quality) : Plutôt que de filtrer manuellement les valeurs aberrantes (ex: températures à $-18.5^\circ\text{C}$ ou humidité à $145\%$), nous utilisons la méthode statistique de l'Intervalle Interquartile (IQR) pour identifier automatiquement les outliers.
- Séries temporelles (Time Series) : Les dates étant initialement stockées au format texte (str), nous les convertissons au type datetime pour extraire des composantes temporelles (heure, jour) et analyser le comportement de consommation électrique selon les plages horaires.  

— Correction préalable : traitement de la colonne pression oubliée

In [64]:
# On avait oublié de traiter les valeurs manquantes de "pression" en Partie 9
print("Valeurs manquantes avant correction :")
print(df.isna().sum())

df["pression"] = df["pression"].fillna(df["pression"].median())

print("\nValeurs manquantes après correction :")
print(df.isna().sum())

Valeurs manquantes avant correction :
id_mesure             0
date_heure            0
id_capteur            0
batiment              0
temperature           0
humidite_relative     0
pression              5
consommation          0
etat                  0
niveau_temperature    0
dtype: int64

Valeurs manquantes après correction :
id_mesure             0
date_heure            0
id_capteur            0
batiment              0
temperature           0
humidite_relative     0
pression              0
consommation          0
etat                  0
niveau_temperature    0
dtype: int64


Conversion en datetime et extraction des composantes temporelles

In [65]:
df["date_heure"] = pd.to_datetime(df["date_heure"])
df["heure"] = df["date_heure"].dt.hour
df["jour"] = df["date_heure"].dt.day_name()

print("Type de la colonne date_heure :", df["date_heure"].dtype)
df[["date_heure", "heure", "jour"]].head()

Type de la colonne date_heure : datetime64[us]


,date_heure,heure,jour
0,2026-01-22 04:00:00,4,Thursday
1,2026-01-17 01:00:00,1,Saturday
2,2026-01-08 04:00:00,4,Thursday
3,2026-01-08 06:00:00,6,Thursday
4,2026-01-12 14:00:00,14,Monday


Détection automatique des outliers par la méthode IQR (fonction sécurisée)

In [66]:
def detect_outliers_iqr(dataframe, colonne):
    """
    Détecte les valeurs aberrantes d'une colonne numérique via la méthode
    de l'intervalle interquartile (IQR).
    Une valeur est considérée comme outlier si elle sort de l'intervalle
    [Q1 - 1.5*IQR ; Q3 + 1.5*IQR].
    """
    # Sécurité : la colonne doit exister et être numérique
    if colonne not in dataframe.columns:
        raise ValueError(f"La colonne '{colonne}' n'existe pas dans le DataFrame.")
    if not pd.api.types.is_numeric_dtype(dataframe[colonne]):
        raise TypeError(f"La colonne '{colonne}' n'est pas numérique.")

    serie = dataframe[colonne].dropna()  # on ignore les NaN restants par sécurité

    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1

    borne_basse = q1 - 1.5 * iqr
    borne_haute = q3 + 1.5 * iqr

    masque_outliers = (dataframe[colonne] < borne_basse) | (dataframe[colonne] > borne_haute)

    return masque_outliers, borne_basse, borne_haute


# Colonnes numériques réellement présentes dans df à ce stade
colonnes_numeriques = ["temperature", "humidite_relative", "pression", "consommation"]

resume_outliers = []
for col in colonnes_numeriques:
    masque, b_basse, b_haute = detect_outliers_iqr(df, col)
    nb_outliers = int(masque.sum())
    resume_outliers.append({
        "colonne": col,
        "borne_basse": round(b_basse, 2),
        "borne_haute": round(b_haute, 2),
        "nb_outliers": nb_outliers,
        "pourcentage": round(nb_outliers / len(df) * 100, 2)
    })

df_resume_outliers = pd.DataFrame(resume_outliers)
print(df_resume_outliers)

             colonne  borne_basse  borne_haute  nb_outliers  pourcentage
0        temperature        15.64        34.25            5         0.83
1  humidite_relative        38.18        91.49            9         1.50
2           pression       990.72      1034.08            5         0.83
3       consommation        21.74       392.91            4         0.67


Isolation des lignes contenant des outliers

In [67]:
masque_global = pd.Series(False, index=df.index)

for col in colonnes_numeriques:
    masque, _, _ = detect_outliers_iqr(df, col)
    masque_global = masque_global | masque

df_outliers = df[masque_global]

print(f"Nombre total de lignes avec au moins une valeur aberrante : {len(df_outliers)}")
df_outliers[["id_mesure", "batiment"] + colonnes_numeriques]

Nombre total de lignes avec au moins une valeur aberrante : 22


,id_mesure,batiment,temperature,humidite_relative,pression,consommation
34,M0212,B003,23.39,68.65,1038.43,188.31
43,M0575,B004,34.72,57.50,1008.94,266.22
52,M0542,B001,25.81,77.47,1035.50,103.08
154,M0335,B004,28.41,71.58,1034.69,270.38
240,M0279,B001,20.54,62.60,850.00,183.55
243,M0416,B003,30.16,54.00,1012.18,875.00
245,M0127,B003,14.11,145.00,1032.36,169.57
258,M0223,B003,15.49,47.96,1006.34,124.90
275,M0536,B003,22.72,61.68,1034.63,167.74
284,M0443,B004,24.25,93.10,1009.39,283.24


Analyse de la consommation par plage horaire

In [68]:
def plage_horaire(heure):
    if 6 <= heure < 12:
        return "Matin"
    elif 12 <= heure < 18:
        return "Après-midi"
    elif 18 <= heure < 22:
        return "Soirée"
    else:
        return "Nuit"

df["plage_horaire"] = df["heure"].apply(plage_horaire)

consommation_par_plage = (
    df.groupby("plage_horaire")["consommation"]
    .agg(["mean", "max", "count"])
    .rename(columns={"mean": "consommation_moyenne", "max": "consommation_max", "count": "nb_mesures"})
    .sort_values("consommation_moyenne", ascending=False)
)

print(consommation_par_plage)

               consommation_moyenne  consommation_max  nb_mesures
plage_horaire                                                    
Nuit                     221.819050            396.90         200
Matin                    219.866867            875.00         150
Après-midi               196.859800            330.34         150
Soirée                   183.304500            384.74         100


**Bilan** de l'analyse bonus :
- Efficacité du filtrage IQR : La méthode IQR a automatiquement ciblé les valeurs extrêmes (captures défaillantes / erreurs de transmission) sans nécessiter de seuils codés en dur (hardcodés).

- Valorisable pour la gestion d'énergie : La conversion en datetime permet désormais de planifier des politiques d'effacement de consommation ou d'optimisation énergétique basées sur les heures de pointe identifiées.

## Conclusion générale — Atelier Pandas (IoT)

Cet atelier a permis de parcourir l'ensemble du cycle de traitement d'un jeu de données réel, des bases de Pandas jusqu'à une analyse proche des standards professionnels. Les Series et DataFrames (Parties 1-2) ont posé les fondations, avant l'exploration (Partie 3) et la sélection/manipulation de colonnes (Parties 4-5) qui ont permis de se familiariser avec la structure des données de capteurs (température, humidité, pression, consommation, état).

Le filtrage et le tri (Parties 6-7) ont ensuite servi à isoler les mesures critiques (températures élevées, combinaisons température/humidité), tandis que les agrégations par bâtiment (Partie 8) ont révélé le bâtiment B004 comme le plus énergivore et le plus sujet aux alertes.

L'étape la plus déterminante pour la fiabilité de l'analyse a été le traitement de la qualité des données (Parties 9-10) : détection et imputation des valeurs manquantes (médiane privilégiée à la moyenne pour rester robuste aux valeurs extrêmes), suppression des doublons, ramenant le jeu de données de 605 à 600 lignes propres. C'est aussi dans cette étape qu'un oubli a été identifié et corrigé — la colonne pression, qui avait échappé au nettoyage initial.

La synthèse statistique (Partie 11) a confirmé une température moyenne autour de 25°C avec des extrêmes suspects (-18,5°C, 58,7°C), et un état des capteurs très majoritairement OK (562/600). Ces données nettoyées ont enfin été exportées en CSV et JSON (Partie 12), prêtes pour un pipeline ML.

Le Bonus (Partie 13) a complété l'atelier avec une dimension plus avancée : conversion temporelle pour analyser la consommation par plage horaire, et détection automatique des anomalies par la méthode IQR — une approche statistique bien plus robuste et généralisable qu'un filtrage manuel des seuils.

En résumé : cet atelier illustre un pipeline complet et réaliste — exploration → nettoyage → analyse → détection automatique d'anomalies → export — directement réutilisable comme brique de prétraitement pour un système IoT en production ou un projet de Machine Learning.